# Document search: retrieval + cited answers

Search a library of mixed-format documents. Anthropic has no embeddings endpoint, so retrieval
uses **Voyage AI** (the provider Anthropic recommends); **Claude** then answers from the top
passages with **citations** pointing back to the exact source.

    ingest (PDF / Word / text / md / CSV) → chunk → embed (Voyage)
      → embed query, rank by cosine → top-k → Claude answer + citations

**Requirements:** `ANTHROPIC_API_KEY` and `VOYAGE_API_KEY` (env vars or a `.env` at the repo root).

In [ ]:
# Setup
import os
import sys

from dotenv import load_dotenv
from anthropic import Anthropic

for _p in (".", "documents"):
    if os.path.isfile(os.path.join(_p, "_documents.py")) and _p not in sys.path:
        sys.path.insert(0, _p)

from _documents import (
    answer,
    answer_text,
    build_library,
    render_html,
    retrieve,
    text_blocks,
)

load_dotenv()
client = Anthropic()

## 1. Build a library

`build_library` takes `(filename, bytes)` pairs, extracts text by file type (PDF via pypdf, Word
via python-docx, text/markdown/CSV decoded directly), splits each into overlapping chunks, and
embeds every chunk with Voyage. Here we use a couple of in-memory documents; point it at real
files by reading their bytes (`open(path, "rb").read()`).

In [ ]:
fox = (
    "The red fox (Vulpes vulpes) is the most widely distributed wild carnivore. "
    "It is an opportunistic omnivore, eating rodents, rabbits, birds, insects, and fruit. "
    "It lives everywhere from arctic tundra to city centers."
).encode("utf-8")

bear = (
    "The polar bear (Ursus maritimus) lives on Arctic sea ice. "
    "It feeds almost exclusively on seals hunted from the ice edge. "
    "The polar bear depends on sea ice to hunt and is vulnerable to its loss."
).encode("utf-8")

chunks, skipped = build_library([("foxes.txt", fox), ("polar_bears.txt", bear)])
print(f"{len(chunks)} chunks from {sorted({c['source'] for c in chunks})}; skipped: {skipped}")

## 2. Retrieve the most relevant passages

The query is embedded as a `query` (Voyage uses a different prompt for queries vs documents),
then ranked against the chunk vectors by cosine similarity. This ranked list is the
"search results."

In [ ]:
query = "Which animal depends on sea ice, and what does it eat?"
retrieved = retrieve(query, chunks, k=3)
for i, p in enumerate(retrieved):
    print(f"#{i + 1}  [{p['source']}]  cosine {p['score']:.3f}")
    print(f"    {p['text'][:90]}…")

## 3. Answer with citations

The retrieved passages are sent to Claude as one **custom-content document** with citations
enabled — each passage is a content block, so citations come back as `content_block_location`
with a block index that maps straight to the retrieved passage. The answer cites which passage
supports each claim.

In [ ]:
content = answer(client, query, retrieved)
print(answer_text(content), "\n")

for block in text_blocks(content):
    for cit in block["citations"]:
        idx = cit["start_block_index"]
        print(f"  cited passage #{idx + 1} ({retrieved[idx]['source']}): {cit['cited_text'][:70]!r}")

## 4. Render it

`render_html` produces a footnoted page: the cited answer with hover-preview markers, plus the
ranked passages panel (click a marker to jump to its passage).

In [ ]:
from IPython.display import HTML

HTML(render_html(query, content, retrieved))

## Notes & next steps

- **Why Voyage:** the Anthropic API has no embeddings endpoint; Voyage is the recommended
  provider. Swap `EMBED_MODEL` in `_documents.py` for a different Voyage model, or replace
  `embed_texts` to use another provider.
- **Chunking** is sentence-aware with a small overlap (`chunk_text`). For transcripts or tables,
  tune `target_chars` / `overlap_chars`, or pre-split your own way.
- **Scale:** for a large corpus, persist the embeddings (e.g. in a vector DB) instead of
  re-embedding each run, and retrieve against the stored index. The retrieval + answer steps stay
  the same.
- **Citations are incompatible with Structured Outputs** — don't combine this with
  `output_config.format` (it 400s).